# Silver Layer — Cleaned and Typed Olist Tables

Takes the 9 raw bronze tables and produces cleaned, properly typed,
deduplicated versions ready for the Gold dimensional model.

**Input:** `workspace.bronze.*`
**Output:** `workspace.silver.*`

## Setup

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

BRONZE = "workspace.bronze"
SILVER = "workspace.silver"


def read_bronze(table_name: str) -> DataFrame:
    return spark.table(f"{BRONZE}.{table_name}")


def write_silver(df: DataFrame, table_name: str) -> None:
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.{table_name}")
    print(f"wrote {table_name}: {df.count()} rows")

## Customers

Just type cleanup here — zip codes stay as strings since they can have
leading zeros, and we trim city/state text.

In [0]:
customers = (
    read_bronze("customers")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast("string"))
    .withColumn("customer_city", F.trim(F.lower(F.col("customer_city"))))
    .withColumn("customer_state", F.trim(F.upper(F.col("customer_state"))))
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    )
)

write_silver(customers, "customers")

## Geolocation

This one's messy - the raw file has dozens of near-duplicate lat/lng
rows per zip prefix (multiple deliveries logged at slightly different
coordinates). We only need one representative point per zip, so we
average them down to a single row.

In [0]:
geolocation = (
    read_bronze("geolocation")
    .withColumn("geolocation_zip_code_prefix", F.col("geolocation_zip_code_prefix").cast("string"))
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("avg_lat"),
        F.avg("geolocation_lng").alias("avg_lng"),
        F.first("geolocation_city").alias("city"),
        F.first("geolocation_state").alias("state"),
    )
)

write_silver(geolocation, "geolocation")

## Products

Joins in the English category name (the raw category is Portuguese)
and fills missing categories instead of dropping those rows - a
product with no category is still a real product.

In [0]:
category_translation = read_bronze("category_translation")

products = (
    read_bronze("products")
    .dropDuplicates(["product_id"])
    .join(category_translation, on="product_category_name", how="left")
    .withColumn(
        "product_category_name_english",
        F.coalesce(F.col("product_category_name_english"), F.lit("unknown")),
    )
    .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))
    .select(
        "product_id",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
)

write_silver(products, "products")

## Sellers

In [0]:
sellers = (
    read_bronze("sellers")
    .dropDuplicates(["seller_id"])
    .withColumn("seller_zip_code_prefix", F.col("seller_zip_code_prefix").cast("string"))
    .withColumn("seller_city", F.trim(F.lower(F.col("seller_city"))))
    .withColumn("seller_state", F.trim(F.upper(F.col("seller_state"))))
)

write_silver(sellers, "sellers")

## Orders

This is the big one - proper timestamp typing plus the delivery-day
and on-time calculations the whole dashboard depends on. Some orders
were never delivered, so delivery_days comes out null for those
instead of erroring - we don't want to silently drop undelivered
orders, they're still real orders.

In [0]:
orders = (
    read_bronze("orders")
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn(
        "delivery_days",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp"),
    )
    .withColumn(
        "is_on_time",
        F.when(F.col("order_delivered_customer_date").isNull(), None)
        .otherwise(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date")),
    )
)

write_silver(orders, "orders")

## Order items

In [0]:
order_items = (
    read_bronze("order_items")
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
)

write_silver(order_items, "order_items")

## Order payments

An order can have several payment rows (installments, or split across
methods). We keep every row here in silver - collapsing to "one
payment method per order" is a gold-layer decision, not a cleaning one.

In [0]:
order_payments = (
    read_bronze("order_payments")
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("payment_value", F.col("payment_value").cast("decimal(10,2)"))
)

write_silver(order_payments, "order_payments")

## Order reviews

A handful of duplicate review_id values show up in the raw file -
keeping the most recent one per review.

In [0]:
from pyspark.sql.window import Window

review_window = Window.partitionBy("review_id").orderBy(F.col("review_creation_date").desc())

order_reviews = (
    read_bronze("order_reviews")
    .withColumn("review_score", F.col("review_score").cast("int"))
    .withColumn("row_num", F.row_number().over(review_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

write_silver(order_reviews, "order_reviews")

## Sanity check

In [0]:
display(spark.sql("SHOW TABLES IN workspace.silver"))